# Mini-Project 5: Taxi Route Revenue, Pricing & Fleet Planner

This analysis uses the ten passenger-count observations in the assignment. Kampala-Ntinda counts are from the original question; Entebbe and Mukono counts are illustrative. Fares and passenger counts are treated as one-way values. Forecasts and fleet sizes are planning estimates based only on this short sample, not a demand guarantee.

In [ ]:
import numpy as np
from pprint import pprint

from src.taxi_planner import (
    DEFAULT_ALPHA_GRID,
    Route,
    plot_route_revenue_forecasts,
    rolling_origin_backtest,
    seasonal_backtest_comparison,
    simulate_weekly_demand,
    solve_ntinda_equilibrium,
    vehicles_required,
)

routes = [
    Route("Kampala-Ntinda", [35, 40, 42, 50, 55, 60, 48, 52, 47, 45], 2_000),
    Route("Kampala-Entebbe", [60, 58, 65, 70, 72, 80, 75, 68, 66, 64], 5_000),
    Route("Kampala-Mukono", [45, 47, 50, 49, 55, 62, 58, 53, 51, 50], 3_000),
]

route_summary = [
    [
        {
            "route": route.name,
            "fare_UGX": route.fare_ugx,
            "mean_passengers": route.descriptive_statistics()["mean"],
            "sample_variance": route.descriptive_statistics()["variance"],
            "sample_std_dev": route.descriptive_statistics()["standard_deviation"],
            "total_revenue_UGX": route.total_revenue(),
        }
        for route in routes
    ]
]
pprint(route_summary)

## Supply-demand equilibrium: Kampala-Ntinda

Equilibrium solves `Qd = 120 - 0.02P` and `Qs = 10 + 0.03P`, with P in UGX and Q in passengers per trip-hour. The supplied fare of UGX 2,000 is below the model equilibrium. Under these assumed straight-line curves that means quantity demanded exceeds quantity supplied at the current price. This is an illustrative model: its coefficients, units, and market conditions should be validated before use in actual fare decisions.

In [ ]:
equilibrium_fare, equilibrium_quantity = solve_ntinda_equilibrium()
current_fare = routes[0].fare_ugx
print(f"Equilibrium fare: UGX {equilibrium_fare:,.0f} per passenger")
print(f"Equilibrium quantity: {equilibrium_quantity:.0f} passengers per trip-hour")
print(f"Current fare UGX {current_fare:,.0f} is below equilibrium by UGX {equilibrium_fare-current_fare:,.0f}.")
assert current_fare < equilibrium_fare

## Rolling-origin forecast comparison

For each route, the backtest makes one-step-ahead forecasts for days 4-10. At each origin, only earlier observations are used. The 3-day moving average and linear trend are compared with simple exponential smoothing. Smoothing alpha is selected from the grid below by its rolling-origin MAE, separately per route. The selected alpha is therefore an in-sample tuning result on this small backtest, not an independent validation score.

In [ ]:
results = {route.name: rolling_origin_backtest(route) for route in routes}
mae_table = [
    [
        {
            "route": route.name,
            **results[route.name].mae_by_model,
            "tuned_alpha": results[route.name].best_alpha,
        }
        for route in routes
    ]
]
pprint(mae_table)

forecast_table = [
    [
        {
            "route": route.name,
            "selected_model": results[route.name].best_model,
            "day11_passengers": results[route.name].forecast_day11_passengers,
            "day11_revenue_UGX": results[route.name].forecast_day11_revenue_ugx,
        }
        for route in routes
    ]
]
pprint(forecast_table)

revenue_figure, revenue_axes = plot_route_revenue_forecasts(routes, results)

## Day-11 fleet plan

One vehicle provides 8 one-way trips x 14 seats = 112 passenger places per day. The plan adds 15% to forecast demand and rounds up to a whole vehicle, avoiding planned capacity below the buffered estimate. It does not account for route timing, load imbalance, return-leg occupancy, downtime, drivers, or vehicle availability.

In [ ]:
fleet_table = [
    [
        {
            "route": route.name,
            "forecast_passengers": results[route.name].forecast_day11_passengers,
            "buffered_passengers": results[route.name].forecast_day11_passengers * 1.15,
            "vehicle_capacity_per_day": 14 * 8,
            "vehicles": vehicles_required(results[route.name].forecast_day11_passengers),
        }
        for route in routes
    ]
]
pprint(fleet_table)

## Extension: weekly demand pattern

A reproducible 60-day synthetic series applies a Friday demand peak, a quieter Sunday, a small upward drift, and random noise. Rolling one-day-ahead scores are compared over the last 21 days. Seasonal-naive forecasts use the count from the same weekday one week earlier; the 3-day moving average cannot represent the weekly cycle as directly.

In [ ]:
weekly_demand = simulate_weekly_demand(days=60, seed=8)
weekly_scores = seasonal_backtest_comparison(weekly_demand, evaluation_days=21)
print("MAE passengers per day:")
pprint(weekly_scores)
assert weekly_scores["seasonal naive"] < weekly_scores["3-day moving average"]
print("Seasonal-naive beats the moving average on this seeded seasonal simulation.")